# Cohort: cardiac surgery x Fitbit

Defines the analysis cohort — participants with EHR data **and** Fitbit data who have a procedure
that is a descendant of OMOP concept `4301351`. Generated from the Verily Workbench cohort
builder (cohort `CardiacSurgxFitbit`, dataset `AllOfUsControlledTierDatasetv9`,
built `20260909_082818`), then adapted to this repo's conventions.

**Governance.** Outputs are stripped on commit, so anything rendered here stays local. Even so,
prefer aggregates over record listings — see the `allofus-data-governance` skill. Figures belong
in `cardiac_surg_fitbit_figures/`; query results are cached in `_cache/`. Both are git-ignored.


In [ ]:
import hashlib
import os
from pathlib import Path

import pandas
import pandas_gbq

# --- Dataset resolution -------------------------------------------------------
# Resolved from the workspace environment rather than hard-coded, so this notebook
# survives a CDR release without edits. Resolved values are printed below and feed
# the cache key, so a new release re-queries instead of serving a stale extract.

_CDR_ENV_CANDIDATES = ("WORKSPACE_CDR", "CDR_DATASET", "WORKSPACE_CDR_DATASET")


def _resolve_cdr():
    for var in _CDR_ENV_CANDIDATES:
        value = os.environ.get(var)
        if value:
            return var, value
    raise RuntimeError(
        "Could not resolve the CDR dataset from the environment. Tried: "
        + ", ".join(_CDR_ENV_CANDIDATES)
        + ". Set the variable this workspace uses (check os.environ for a key "
          "containing 'CDR'), or export it before starting the kernel. "
          "Do not hard-code the project ID."
    )


CDR_ENV_VAR, CDR = _resolve_cdr()
CDR_PROJECT, _, CDR_DATASET = CDR.partition(".")


def discover_cohort_index(cdr=None, verbose=True):
    """Find the cohort-builder index dataset: the most recently created one.

    The Workbench cohort builder writes a per-build dataset '<CDR>_index_<suffix>'
    holding the T_ENT_* / T_HAD_* / T_ESA_* cohort tables. It has no environment
    variable and changes when the COHORT is rebuilt -- a different trigger from a
    CDR release -- so it is discovered rather than hard-coded, and a rebuild
    therefore changes the cache key and forces a re-query.

    Ordered by creation_time, which is chronologically correct regardless of how
    the name suffix is formatted.
    """
    cdr = cdr or CDR
    project, _, dataset = cdr.partition(".")
    sql = f"""
        SELECT schema_name, creation_time
        FROM `{project}`.INFORMATION_SCHEMA.SCHEMATA
        WHERE STARTS_WITH(schema_name, '{dataset}_index_')
        ORDER BY creation_time DESC
    """
    candidates = pandas_gbq.read_gbq(sql, dialect="standard", progress_bar_type=None)

    if candidates.empty:
        raise RuntimeError(
            f"No cohort index dataset found starting with '{dataset}_index_' in "
            f"project '{project}'. Build the cohort in the Workbench cohort builder, "
            f"or set WORKSPACE_CDR_INDEX explicitly."
        )

    if verbose and len(candidates) > 1:
        print(f"Cohort index: {len(candidates)} builds found, using the newest:")
        for _, row in candidates.iterrows():
            print(f"    {row['schema_name']:42s} created {row['creation_time']}")

    return f"{project}.{candidates.iloc[0]['schema_name']}"


# An explicit override wins; otherwise discover the newest build.
CDR_INDEX = os.environ.get("WORKSPACE_CDR_INDEX") or discover_cohort_index()

# --- Paths (both git-ignored) -------------------------------------------------
NOTEBOOK = "cardiac_surg_fitbit"
FIGURES = Path(f"{NOTEBOOK}_figures")
CACHE_DIR = Path("_cache")
FIGURES.mkdir(exist_ok=True)
CACHE_DIR.mkdir(exist_ok=True)

print(f"CDR         : {CDR}   (from ${CDR_ENV_VAR})")
print(f"Cohort index: {CDR_INDEX}")
print(f"Figures     : {FIGURES}/")
print(f"Cache       : {CACHE_DIR}/")


In [ ]:
def cached_gbq(sql, name):
    """Run `sql`, caching the result as Parquet in _cache/.

    The cache key hashes the SQL together with the resolved CDR and cohort-index
    dataset names, so a CDR release or a cohort rebuild misses rather than serving
    a stale extract. Cached files hold row-level data and stay inside the workspace:
    _cache/ is git-ignored and must never be committed or downloaded.
    """
    key = hashlib.sha256("|".join([sql, CDR, CDR_INDEX]).encode()).hexdigest()[:16]
    path = CACHE_DIR / f"{name}_{key}.parquet"

    if path.exists():
        df = pandas.read_parquet(path)
        print(f"[cache hit ] {name}: {len(df):,} rows from {path.name}")
        return df

    print(f"[cache miss] {name}: querying BigQuery ...")
    # Try the BigQuery Storage API for speed; fall back to the REST API if it is
    # not enabled or permissions are missing (e.g. outside the Workbench).
    try:
        df = pandas_gbq.read_gbq(
            sql,
            dialect="standard",
            use_bqstorage_api=True,
            progress_bar_type="tqdm_notebook",
        )
    except Exception:
        df = pandas_gbq.read_gbq(
            sql,
            dialect="standard",
            use_bqstorage_api=False,
            progress_bar_type="tqdm_notebook",
        )

    df.to_parquet(path, index=False)
    print(f"[cached    ] {name}: {len(df):,} rows -> {path.name}")
    return df


In [ ]:
person_sql = f"""

    SELECT
        st.birth_datetime,
        st.gender_concept_id,
        dt0.concept_name AS T_DISP_gender,
        st.person_id,
        st.sex_at_birth_concept_id,
        dt1.concept_name AS T_DISP_sex_at_birth 
    FROM
        `{CDR}`.person AS st 
    LEFT JOIN
        `{CDR}`.concept AS dt0 
            ON dt0.concept_id = st.gender_concept_id 
    LEFT JOIN
        `{CDR}`.concept AS dt1 
            ON dt1.concept_id = st.sex_at_birth_concept_id 
    WHERE
        st.person_id IN (SELECT
            id 
        FROM
            `{CDR_INDEX}`.T_ENT_person 
        WHERE
            (has_ehr_data = true) 
            AND (has_fitbit = true) 
            AND (id IN (SELECT
                person_id AS primary_id 
            FROM
                `{CDR_INDEX}`.T_ENT_procedureOccurrence 
            WHERE
                procedure_concept_id IN (SELECT
                    descendant 
                FROM
                    `{CDR_INDEX}`.T_HAD_procedureConcept_default 
                WHERE
                    ancestor = 4301351 
                UNION
                ALL SELECT
                    4301351))))
"""

person_df = cached_gbq(person_sql, "person")


In [ ]:
procedure_occurrence_sql = f"""

    SELECT
        st.modifier_source_value,
        st.person_id,
        st.procedure_concept_id,
        st.procedure_datetime,
        st.procedure_type_concept_id,
        dt0.concept_name AS T_DISP_procedure_type_concept_name,
        st.procedure_source_concept_id,
        dt1.vocabulary_id AS T_DISP_source_vocabulary,
        dt2.concept_code AS T_DISP_standard_concept_code,
        dt2.concept_name AS T_DISP_standard_concept_name,
        dt2.vocabulary_id AS T_DISP_standard_vocabulary,
        st.visit_occurrence_id 
    FROM
        `{CDR}`.procedure_occurrence AS st 
    LEFT JOIN
        `{CDR}`.concept AS dt0 
            ON dt0.concept_id = st.procedure_type_concept_id 
    LEFT JOIN
        `{CDR}`.concept AS dt1 
            ON dt1.concept_id = st.procedure_source_concept_id 
    LEFT JOIN
        `{CDR}`.concept AS dt2 
            ON dt2.concept_id = st.procedure_concept_id 
    WHERE
        st.procedure_occurrence_id IN (SELECT
            id 
        FROM
            `{CDR_INDEX}`.T_ESA_procedureOccurrence_person_id 
        WHERE
            (person_id IN (SELECT
                id 
            FROM
                `{CDR_INDEX}`.T_ENT_person 
            WHERE
                (has_ehr_data = true) 
                AND (has_fitbit = true) 
                AND (id IN (SELECT
                    person_id AS primary_id 
                FROM
                    `{CDR_INDEX}`.T_ENT_procedureOccurrence 
                WHERE
                    procedure_concept_id IN (SELECT
                        descendant 
                    FROM
                        `{CDR_INDEX}`.T_HAD_procedureConcept_default 
                    WHERE
                        ancestor = 4301351 
                    UNION
                    ALL SELECT
                        4301351))))) 
                    AND (procedure_concept_id IN (SELECT
                        descendant 
                    FROM
                        `{CDR_INDEX}`.T_HAD_procedureConcept_default 
                    WHERE
                        ancestor = 4301351 
                    UNION
                    ALL SELECT
                        4301351)))
"""

procedure_occurrence_df = cached_gbq(procedure_occurrence_sql, "procedure_occurrence")


In [ ]:
# Aggregate summary only — no participant-level record listing.
# The original notebook ended each query with .head(5); eyeballing rows interactively is
# fine, but a listing that lands in saved output is a DUA problem, so the default here is
# shape + schema. Suppress small cells before reporting any count table externally.

for label, df in [("person", person_df), ("procedure_occurrence", procedure_occurrence_df)]:
    print(f"{label:22s} rows={len(df):>9,}  cols={df.shape[1]:>3}  "
          f"unique person_id={df['person_id'].nunique():>8,}")

print()
print("person columns             :", list(person_df.columns))
print("procedure_occurrence columns:", list(procedure_occurrence_df.columns))
